# 1G — Multi-seed pareado: PubMedBERT, BioLinkBERT-base, BioBERT, SciBERT

**Motivo:** `1CF-todos-los-encoders.ipynb` (y los notebooks 1A/1B originales) nunca fijan
seed de verdad -- `"seed": 42` en los `results_*.json` es solo una etiqueta, nunca
siembra `random`/`numpy`/`torch`. Por eso el mismo config de BioLinkBERT-base dio
0.7835 curado / 0.308 ciego en un run y 0.8172 / 0.273 en otro: sin seed real, cada
entrenamiento es una muestra distinta, y no se puede saber si una diferencia entre
encoders es real o ruido.

**Diseño pareado:** las mismas 3 seeds (42, 123, 2024) para los 4 encoders base
(PubMedBERT, BioLinkBERT-base, BioBERT, SciBERT) -- no seeds sueltas por modelo, para
poder comparar manzanas con manzanas en cada seed.

**Por cada (encoder, seed):** entrena con seed fijada de verdad, evalúa en el
protocolo ciego oficial (282k candidatos de `eng_dev_blind.txt`, puntuado con
`baseline/score.py` contra el gold real), y barre el threshold hasta 0.99 (un grid
recortado a 0.95 ya nos dio un óptimo falso -- varios modelos pegados al borde).

**Resumible:** si ya existe `results_seed_summary.json` para esa combinación, la
salta -- si se corta a medias, no repite lo ya hecho.

**Coste:** 3 seeds x 4 encoders = 12 entrenamientos completos (~30-40 min cada uno,
batch=16) + inferencia blind (~20 min cada uno) + barrido de threshold (rápido, sobre
probabilidades ya cacheadas) -- realista, varias horas en total. Pensado para dejarlo
corriendo largo rato (nohup/tmux si se ejecuta desde terminal).

**Decisión metodológica documentada:** el checkpoint de cada (encoder, seed) se
selecciona por el mejor macro F1 en el **dev curado** (igual que en todos los
notebooks anteriores), y *después* se reporta su desempeño en ciego -- no se
selecciona el epoch por la métrica que se termina reportando. Es coherente entre
los 4 encoders (misma regla para todos, comparación justa) pero es una elección
consciente, no la única posible: seleccionar por ciego en cada epoch sería más
"correcto" en teoría pero requiere inferir sobre los 282k candidatos en cada
epoch (carísimo). Esta limitación debe quedar explícita en la memoria.

## 1. Setup

In [ ]:
# Ejecucion en servidor local (zape), entorno conda "tfg".
# Override de la cache de HuggingFace a una carpeta escribible del home.
# EJECUTAR ANTES de cualquier import de opennre/transformers/huggingface_hub.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])

# Parches de compatibilidad de OpenNRE (UTF-8, AdamW de torch, num_workers=0)
!python ../baseline/patch_opennre.py

In [ ]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path
import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_universal_encoder
add_macro_f1_metric()
fix_universal_encoder()  # backbone-agnostic BERTEntityEncoder (permite anadir XLM-RoBERTa)
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2. Configuración global

In [ ]:
# Hiperparametros comunes (identicos a 1A/1B/1CF)
MAX_LENGTH    = 256
LEARNING_RATE = 2e-5
EPOCHS        = 15
WARMUP_STEPS  = 300
NEG_RATIO     = 3
GRAD_CLIP_NORM = 1.0  # evita la divergencia que vimos en modelos large (loss subiendo, dev colapsando)

DATA_DIR    = Path("../data/english")
TRAIN_DATA  = DATA_DIR / "eng_train.txt"
DEV_DATA    = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"
with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print("Clases:", len(rel2id))

SEEDS = [42, 123, 2024]

MULTISEED_CONFIGS = [
    {"exp": "pubmedbert",       "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext", "batch": 16, "base_outdir": "1A-pubmedbert"},
    {"exp": "biolinkbert_base", "model": "michiyasunaga/BioLinkBERT-base",                                 "batch": 16, "base_outdir": "1B-biolinkbert"},
    {"exp": "biobert",          "model": "dmis-lab/biobert-v1.1",                                           "batch": 16, "base_outdir": "1C-biobert"},
    {"exp": "scibert",          "model": "allenai/scibert_scivocab_uncased",                                "batch": 16, "base_outdir": "1D-scibert"},
    {"exp": "xlm_roberta",       "model": "xlm-roberta-base",                                                "batch": 16, "base_outdir": "1H-xlm-roberta"},
]
for c in MULTISEED_CONFIGS:
    print(f"  {c['exp']:<20} {c['model']:<55} batch={c['batch']}")

## 3. Datos del protocolo ciego (blind)

Se cargan una vez y se reutilizan para los 12 (encoder, seed).

In [ ]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

# grid de threshold: el recortado a 0.95 nos dio un optimo falso (varios modelos
# pegados al borde). Se extiende hasta 0.99 y se afina a pasos de 0.01 entre
# 0.81-0.99 (es gratis, reutiliza las probs ya cacheadas) -- los optimos vistos
# hasta ahora (0.90, 0.96, 0.97, 0.98) caen todos en esa zona.
THRESH_GRID = ([round(float(x), 2) for x in np.arange(0.05, 0.801, 0.05)] +
               [round(float(x), 2) for x in np.arange(0.81, 0.991, 0.01)])

def _preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def _rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

## 4. Seed real + bucle de entrenamiento

`set_seed` es lo que faltaba en todo el pipeline hasta ahora: siembra `random`,
`numpy`, `torch` (CPU y CUDA) y fuerza `cudnn.deterministic=True` -- se llama
*antes* de construir encoder/modelo/framework, que es donde se inicializa la
cabeza de clasificación y se baraja el dataset.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    """Reimplementa el bucle de SentenceRE para validar y guardar el mejor
    checkpoint por macro_f1 en cada epoch, devolviendo el historial."""
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history

## 5. Función que entrena + evalúa un (encoder, seed) completo

In [ ]:
def run_one_seed_full(cfg, seed):
    EXP, MODEL, BATCH = cfg["exp"], cfg["model"], cfg["batch"]
    OUT = Path(f"../outputs/{cfg['base_outdir']}/seed{seed}")
    OUT.mkdir(parents=True, exist_ok=True)
    CKPT = OUT / f"eng_{EXP}.pth.tar"
    SUMMARY_PATH = OUT / "results_seed_summary.json"

    if SUMMARY_PATH.exists():
        print(f"[skip] {EXP} seed={seed} ya tiene results_seed_summary.json")
        return json.load(open(SUMMARY_PATH))

    print("\n" + "#" * 70 + f"\n# {EXP}  seed={seed}\n" + "#" * 70)
    set_seed(seed)

    encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL)
    model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
    framework = opennre.framework.SentenceRE(
        model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
        ckpt=str(CKPT), batch_size=BATCH, max_epoch=EPOCHS, lr=LEARNING_RATE,
        opt="adamw", warmup_step=WARMUP_STEPS)

    t0 = time.time()
    history = train_with_history(framework, EPOCHS, metric="macro_f1")
    train_minutes = (time.time() - t0) / 60
    with open(OUT / f"history_{EXP}.json", "w") as f:
        json.dump(history, f, indent=2)
    best = max(history, key=lambda h: h["val_macro_f1"])
    macro_f1_curado = best["val_macro_f1"]

    # recargar el mejor checkpoint para blind + calibracion
    model.load_state_dict(torch.load(str(CKPT), map_location="cpu")["state_dict"])
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device); model.eval()

    t0 = time.time()
    all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(blind_raw), 64):
            batch = blind_raw[s:s + 64]
            tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                     "t": {"pos": i["t"]["pos"]}}) for i in batch]
            fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
            logits = model(*fields)
            all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
    blind_minutes = (time.time() - t0) / 60
    np.save(OUT / f"blind_probs_{EXP}.npy", all_probs)

    argmax_ids = all_probs.argmax(axis=1)
    res_argmax = evaluate(pd.DataFrame(_rows_from_preds(argmax_ids)), gold_df_blind)

    sweep_rows = []
    for th in THRESH_GRID:
        pred_ids = _preds_at_threshold(all_probs, th)
        res = evaluate(pd.DataFrame(_rows_from_preds(pred_ids)), gold_df_blind)
        sweep_rows.append({"threshold": th, "macro_f1": res["macro_f1"]})
    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv(OUT / f"threshold_sweep_{EXP}.csv", index=False)
    best_row = sweep_df.loc[sweep_df["macro_f1"].idxmax()]
    best_threshold = float(best_row["threshold"])
    at_edge = best_threshold == THRESH_GRID[-1]

    summary = {
        "exp": EXP, "model": MODEL, "seed": seed,
        "macro_f1_curado": macro_f1_curado,
        "macro_f1_ciego_argmax": res_argmax["macro_f1"],
        "best_threshold": best_threshold,
        "macro_f1_ciego_calibrado": float(best_row["macro_f1"]),
        "threshold_en_borde_del_grid": at_edge,
        "train_minutes": round(train_minutes, 1),
        "blind_minutes": round(blind_minutes, 1),
    }
    with open(SUMMARY_PATH, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"[{EXP} seed={seed}] curado={macro_f1_curado:.4f}  ciego_argmax={res_argmax['macro_f1']:.4f}  "
          f"ciego_calibrado={summary['macro_f1_ciego_calibrado']:.4f} (th={best_threshold:.2f}"
          f"{', BORDE DEL GRID -- revisar' if at_edge else ''})")

    del framework, model, encoder; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

## 6. Ejecutar las 3 seeds x 4 encoders

Resumible: si se corta a medias, relanzar esta celda no repite lo ya guardado.

In [ ]:
multiseed_summaries = []
for cfg in MULTISEED_CONFIGS:
    for seed in SEEDS:
        multiseed_summaries.append(run_one_seed_full(cfg, seed))

print("\nMULTI-SEED COMPLETADO")

## 7. Agregación: media ± std, ranking seed a seed, diferencias pareadas

Regla: si la diferencia entre dos encoders es mayor que 2-3x la std entre
seeds, se considera real; si es menor, la conclusión honesta es "empatan".

In [ ]:
all_seed_rows = []
for cfg in MULTISEED_CONFIGS:
    for seed in SEEDS:
        p = Path(f"../outputs/{cfg['base_outdir']}/seed{seed}/results_seed_summary.json")
        if p.exists():
            all_seed_rows.append(json.load(open(p)))

df = pd.DataFrame(all_seed_rows)
if df.empty:
    print("Todavia no hay resultados de multi-seed guardados.")
else:
    if df["threshold_en_borde_del_grid"].any():
        print("AVISO: algun best_threshold cayo en el borde del grid (0.99) -- "
              "el verdadero optimo podria estar mas alla, igual que paso antes con 0.95.\n")

    print("=== Media +/- std por encoder (ciego calibrado) ===")
    agg = df.groupby("exp")["macro_f1_ciego_calibrado"].agg(["mean", "std", "count"])
    agg = agg.sort_values("mean", ascending=False)
    print(agg.to_string())

    print("\n=== Ranking seed a seed (ciego calibrado) ===")
    pivot = df.pivot(index="seed", columns="exp", values="macro_f1_ciego_calibrado")
    print(pivot.to_string())
    rankings = pivot.rank(axis=1, ascending=False)
    ranking_stable = (rankings.nunique() == 1).all()
    print(f"\n¿Ranking identico en las {len(SEEDS)} seeds? {'SI' if ranking_stable else 'NO'}")

    print("\n=== Diferencias pareadas por seed (col - fila) ===")
    exps = pivot.columns.tolist()
    for i, a in enumerate(exps):
        for b in exps[i+1:]:
            diffs = pivot[b] - pivot[a]
            signo = "SIEMPRE +" if (diffs > 0).all() else ("SIEMPRE -" if (diffs < 0).all() else "CAMBIA DE SIGNO")
            print(f"{b} - {a}: {diffs.values.round(4).tolist()}  -> {signo}")

    # tambien en curado, para contraste
    print("\n=== Media +/- std por encoder (curado, referencia) ===")
    agg_curado = df.groupby("exp")["macro_f1_curado"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
    print(agg_curado.to_string())

    Path("../outputs/multiseed").mkdir(parents=True, exist_ok=True)
    df.to_csv("../outputs/multiseed/multiseed_resultados.csv", index=False)
    print("\nGuardado: ../outputs/multiseed/multiseed_resultados.csv")